---
# Part 2 — Deep Learning: U-Net Semantic Segmentation

We now implement a **U-Net** — a fully-convolutional encoder-decoder architecture originally designed for biomedical image segmentation but proven highly effective for satellite land-cover mapping.  
The network takes a **(17, H, W)** tensor (12 bands + 5 spectral indices) and produces a **(4, H, W)** class-probability map.

Key design choices:
- **Patch-based training** (128×128 tiles) — avoids GPU OOM on full-scene tensors  
- **Skip connections** — preserve fine spatial detail that deep pooling erases  
- **Class-weighted CrossEntropy** — compensates for Sand/Water pixel imbalance  
- **Unknown-class mask** — class-0 pixels are excluded from loss and metrics  

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2.1 — Patch Dataset

We slice each full scene into **128×128 patches** at training time.  
Patches where the valid (non-zero) pixel ratio is below **15 %** are discarded — they are mostly border artefacts from the GEE export and would confuse the network.

In [ ]:
PATCH_SIZE   = 128
PATCH_STRIDE = 96    # overlap keeps border context
MIN_VALID    = 0.15  # discard patches with < 15 % labeled pixels
IN_CHANNELS  = 17
NUM_CLASSES  = 5     # 0-Unknown, 1-Greenery, 2-Sand, 3-Water, 4-Cement

class PatchDataset(Dataset):
    """
    Slices every scene in `location_list` into (PATCH_SIZE × PATCH_SIZE) patches.
    Patches dominated by unknown pixels are skipped.
    """
    def __init__(self, location_list, augment=False):
        self.patches = []    # list of (features_patch, mask_patch)
        self.augment = augment
        self._build(location_list)

    def _build(self, location_list):
        for loc in location_list:
            try:
                features, mask = load_image_and_mask(loc)  # reuse function from Part 1
                H, W = mask.shape
                for r in range(0, H - PATCH_SIZE + 1, PATCH_STRIDE):
                    for c in range(0, W - PATCH_SIZE + 1, PATCH_STRIDE):
                        m_patch = mask[r:r+PATCH_SIZE, c:c+PATCH_SIZE]
                        valid_ratio = (m_patch > 0).mean()
                        if valid_ratio < MIN_VALID:
                            continue
                        f_patch = features[:, r:r+PATCH_SIZE, c:c+PATCH_SIZE]
                        self.patches.append((
                            f_patch.astype(np.float32),
                            m_patch.astype(np.int64)
                        ))
            except Exception as e:
                print(f'  skipping {loc}: {e}')
        print(f'  built {len(self.patches):,} patches from {len(location_list)} scenes')

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        x, y = self.patches[idx]
        x = torch.from_numpy(x)
        y = torch.from_numpy(y)
        if self.augment:
            # random horizontal / vertical flip — free augmentation for satellite images
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[2])
                y = torch.flip(y, dims=[1])
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[1])
                y = torch.flip(y, dims=[0])
            # random 90° rotation
            k = torch.randint(0, 4, (1,)).item()
            x = torch.rot90(x, k, dims=[1, 2])
            y = torch.rot90(y, k, dims=[0, 1])
        return x, y

print('building training dataset …')
train_ds = PatchDataset(TRAIN_LOCS, augment=True)
print('building validation dataset …')
val_ds   = PatchDataset(VAL_LOCS,   augment=False)
print('building test dataset …')
test_ds  = PatchDataset(TEST_LOCS,  augment=False)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'\nTrain patches : {len(train_ds):,}')
print(f'Val   patches : {len(val_ds):,}')
print(f'Test  patches : {len(test_ds):,}')

## 2.2 — U-Net Architecture

```
Input (17 ch)
  │
  ▼  Encoder
  E1 (64)  ──────────────────────────────────── skip ──┐
    MaxPool                                             │
  E2 (128) ─────────────────────────────── skip ──┐    │
    MaxPool                                        │    │
  E3 (256) ────────────────────── skip ──┐         │    │
    MaxPool                              │         │    │
  E4 (512) ─────────── skip ──┐          │         │    │
    MaxPool                   │          │         │    │
  Bottleneck (1024)           │          │         │    │
  Decoder                     │          │         │    │
  D4 (512) ◄──────────────────┘          │         │    │
  D3 (256) ◄───────────────────────────-┘         │    │
  D2 (128) ◄──────────────────────────────────────┘    │
  D1 (64)  ◄─────────────────────────────────────────--┘
  Output (NUM_CLASSES)
```

In [ ]:
def double_conv(in_ch, out_ch):
    """Two successive 3×3 convolutions with BatchNorm and ReLU."""
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, in_channels=17, num_classes=5, base_filters=64):
        super().__init__()
        f = base_filters

        # ── Encoder ──────────────────────────────────────
        self.enc1 = double_conv(in_channels, f)        # → (f,   H,   W)
        self.enc2 = double_conv(f,     f * 2)          # → (2f,  H/2, W/2)
        self.enc3 = double_conv(f * 2, f * 4)          # → (4f,  H/4, W/4)
        self.enc4 = double_conv(f * 4, f * 8)          # → (8f,  H/8, W/8)

        self.pool = nn.MaxPool2d(2)

        # ── Bottleneck ───────────────────────────────────
        self.bottleneck = double_conv(f * 8, f * 16)   # → (16f, H/16, W/16)

        # ── Decoder ──────────────────────────────────────
        self.up4   = nn.ConvTranspose2d(f * 16, f * 8,  kernel_size=2, stride=2)
        self.dec4  = double_conv(f * 16, f * 8)

        self.up3   = nn.ConvTranspose2d(f * 8,  f * 4,  kernel_size=2, stride=2)
        self.dec3  = double_conv(f * 8,  f * 4)

        self.up2   = nn.ConvTranspose2d(f * 4,  f * 2,  kernel_size=2, stride=2)
        self.dec2  = double_conv(f * 4,  f * 2)

        self.up1   = nn.ConvTranspose2d(f * 2,  f,      kernel_size=2, stride=2)
        self.dec1  = double_conv(f * 2,  f)

        # ── Output head ──────────────────────────────────
        self.out_conv = nn.Conv2d(f, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))

        # Bottleneck
        bn = self.bottleneck(self.pool(s4))

        # Decoder — upsample then concatenate the skip connection
        x = self.dec4(torch.cat([self.up4(bn), s4], dim=1))
        x = self.dec3(torch.cat([self.up3(x),  s3], dim=1))
        x = self.dec2(torch.cat([self.up2(x),  s2], dim=1))
        x = self.dec1(torch.cat([self.up1(x),  s1], dim=1))

        return self.out_conv(x)   # raw logits: (B, NUM_CLASSES, H, W)


model = UNet(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'U-Net parameters: {total_params:,}')
print(model)

## 2.3 — Loss Function & Optimizer

**Class weights** correct for imbalanced pixel counts (desert Sand dominates Egyptian scenes).  
**Unknown (class 0)** is excluded by setting its weight to 0 and using `ignore_index=0`.

In [ ]:
# Estimate class frequencies from the training pixel set y_train (Part-1 variable)
class_counts = np.array([(y_train == c).sum() for c in range(NUM_CLASSES)], dtype=np.float64)
class_counts[0] = 1.0   # prevent div-by-zero; weight will be set to 0 anyway

# Inverse-frequency weights; Unknown class gets 0
weights = 1.0 / class_counts
weights[0] = 0.0
weights = weights / weights[1:].sum()  # normalise over the 4 active classes
weights_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print('class weights:')
all_names = ['Unknown', 'Greenery', 'Sand', 'Water', 'Cement']
for n, w in zip(all_names, weights):
    print(f'  {n:<12}: {w:.6f}')

criterion = nn.CrossEntropyLoss(weight=weights_tensor, ignore_index=0)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# Cosine annealing — gently reduces LR towards zero over training
NUM_EPOCHS = 30
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

## 2.4 — Training Loop

In [ ]:
def compute_patch_miou(preds, targets):
    """
    Vectorised mIoU over a batch of predicted and target tensors.
    Unknown (class 0) is excluded.
    preds  : (N,) int64 flattened predictions
    targets: (N,) int64 flattened ground truth
    """
    iou_list = []
    for c in CLASS_LABELS:            # CLASS_LABELS = [1,2,3,4] from Part 1
        pred_c   = (preds   == c)
        target_c = (targets == c)
        intersection = (pred_c & target_c).sum()
        union        = (pred_c | target_c).sum()
        if union == 0:
            continue
        iou_list.append(intersection / union)
    return float(np.mean(iou_list)) if iou_list else 0.0


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_miou, n_batches = 0.0, 0.0, 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            logits = model(X_batch)           # (B, 5, 128, 128)
            loss   = criterion(logits, y_batch)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            preds   = logits.argmax(dim=1).cpu().numpy().flatten()
            targets = y_batch.cpu().numpy().flatten()

            # exclude unknown pixels from metric
            valid = targets > 0
            batch_miou = compute_patch_miou(preds[valid], targets[valid])

            total_loss  += loss.item()
            total_miou  += batch_miou
            n_batches   += 1

    return total_loss / n_batches, total_miou / n_batches


BEST_MODEL_PATH = os.path.join(MODEL_FOLDER, 'unet_best.pth')

history = {'train_loss': [], 'val_loss': [], 'train_miou': [], 'val_miou': []}
best_val_miou = 0.0

print(f'Training U-Net for {NUM_EPOCHS} epochs on {DEVICE} …')
print(f'{"Epoch":>5}  {"Train Loss":>10}  {"Val Loss":>10}  {"Train mIoU":>11}  {"Val mIoU":>10}')

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_miou = run_epoch(train_loader, train=True)
    vl_loss, vl_miou = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_miou'].append(tr_miou)
    history['val_miou'].append(vl_miou)

    flag = ''
    if vl_miou > best_val_miou:
        best_val_miou = vl_miou
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        flag = '  ← best'

    print(f'{epoch:>5}  {tr_loss:>10.4f}  {vl_loss:>10.4f}  {tr_miou:>11.4f}  {vl_miou:>10.4f}{flag}')

print(f'\nBest val mIoU: {best_val_miou:.4f}')

## 2.5 — Training Curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training & Validation Loss'); axes[0].legend()

axes[1].plot(epochs, history['train_miou'], label='Train')
axes[1].plot(epochs, history['val_miou'],   label='Val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU')
axes[1].set_title('Training & Validation mIoU'); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(MODEL_FOLDER, 'unet_training_curves.png'), dpi=150)
plt.show()

## 2.6 — Full Evaluation on Test Set

We reload the best checkpoint and run pixel-level evaluation across all test scenes.

In [ ]:
def evaluate_unet(loader, split_name):
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    model.eval()

    all_preds, all_targets = [], []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            logits = model(X_batch.to(DEVICE))
            preds  = logits.argmax(dim=1).cpu().numpy().flatten()
            targets = y_batch.numpy().flatten()

            valid = targets > 0
            all_preds.append(preds[valid])
            all_targets.append(targets[valid])

    all_preds   = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    # Error matrix
    cm = confusion_matrix(all_targets, all_preds, labels=CLASS_LABELS)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Greens')
    ax.set_title(f'U-Net Error Matrix: {split_name}')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_FOLDER, f'unet_error_matrix_{split_name}.png'), dpi=150)
    plt.show()

    print(f'Errors of omission (missed true pixels):')
    for i, name in enumerate(CLASS_NAMES):
        err = 1 - cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0
        print(f'  {name}: {err:.4f}')

    print(f'\nErrors of commission (false positives):')
    for i, name in enumerate(CLASS_NAMES):
        err = 1 - cm[i, i] / cm[:, i].sum() if cm[:, i].sum() > 0 else 0
        print(f'  {name}: {err:.4f}')

    iou_dict, miou = compute_miou(all_targets, all_preds)   # reuse Part-1 function
    print(f'\nmIoU: {miou:.4f}')
    for label, name in zip(CLASS_LABELS, CLASS_NAMES):
        print(f'  {name}: {iou_dict[label]:.4f}')

    print(classification_report(all_targets, all_preds, target_names=CLASS_NAMES, labels=CLASS_LABELS))
    return miou, iou_dict


print('=== Validation set ===')
unet_val_miou, unet_val_iou = evaluate_unet(val_loader, 'Validation')

print('\n=== Test set ===')
unet_test_miou, unet_test_iou = evaluate_unet(test_loader, 'Test')

## 2.7 — Thematic Maps: U-Net Predictions vs Ground Truth

We run the best checkpoint over the full-resolution test scenes (not patches) using a sliding-window inference to handle arbitrary image sizes without padding artefacts.

In [ ]:
def sliding_window_predict(features_np, patch_size=128, stride=64):
    """
    Run the U-Net over a full scene using a sliding window.
    Overlapping predictions are averaged before argmax.
    features_np : (17, H, W) float32
    """
    model.eval()
    _, H, W = features_np.shape
    logit_sum   = np.zeros((NUM_CLASSES, H, W), dtype=np.float32)
    count_map   = np.zeros((H, W),              dtype=np.float32)

    with torch.no_grad():
        for r in range(0, H - patch_size + 1, stride):
            for c in range(0, W - patch_size + 1, stride):
                patch = features_np[:, r:r+patch_size, c:c+patch_size]
                t     = torch.from_numpy(patch).unsqueeze(0).to(DEVICE)   # (1,17,P,P)
                logit = model(t).squeeze(0).cpu().numpy()                  # (5,P,P)
                logit_sum[:, r:r+patch_size, c:c+patch_size] += logit
                count_map[r:r+patch_size, c:c+patch_size]    += 1.0

    count_map = np.maximum(count_map, 1)                # avoid div-by-zero at borders
    avg_logit = logit_sum / count_map[np.newaxis, :, :]
    return avg_logit.argmax(axis=0)                     # (H, W) class map


def make_unet_thematic_map(location_name):
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))

    features, mask = load_image_and_mask(location_name)
    pred_img = sliding_window_predict(features.astype(np.float32))

    # keep unknown border pixels as 0
    pred_img[mask == 0] = 0

    colors = ['black', 'darkgreen', 'yellow', 'darkblue', 'gray']
    cmap   = mcolors.ListedColormap(colors)
    norm   = mcolors.BoundaryNorm([0, 1, 2, 3, 4, 5], cmap.N)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(mask,     cmap=cmap, norm=norm)
    axes[0].set_title('Ground Truth');     axes[0].axis('off')

    im = axes[1].imshow(pred_img, cmap=cmap, norm=norm)
    axes[1].set_title('U-Net Prediction'); axes[1].axis('off')

    cbar = plt.colorbar(im, ax=axes, ticks=[0.5,1.5,2.5,3.5,4.5],
                        fraction=0.02, pad=0.04)
    cbar.ax.set_yticklabels(['Unknown','Greenery','Sand','Water','Cement'])

    plt.suptitle(f'U-Net Land Cover Map — {location_name}')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_FOLDER, f'{location_name}_unet_prediction.png'), dpi=150)
    plt.show()

for loc in TEST_LOCS[:3]:
    make_unet_thematic_map(loc)

---
# Part 3 — Discussion & Comparative Analysis

This section critically compares the **Minimum Distance Classifier (MDC)** against the **U-Net** across four axes: performance, computation, design reasoning, and edge cases.

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────
print('=' * 65)
print(f'{"Metric":<30} {"MDC":>15} {"U-Net":>15}')
print('=' * 65)

rows = [
    ('Val  mIoU',       f'{val_miou:.4f}',       f'{unet_val_miou:.4f}'),
    ('Test mIoU',       f'{test_miou:.4f}',       f'{unet_test_miou:.4f}'),
]
for label, name in zip(CLASS_LABELS, CLASS_NAMES):
    rows.append((f'  Test IoU — {name}',
                 f'{test_iou[label]:.4f}',
                 f'{unet_test_iou[label]:.4f}'))

rows += [
    ('Trainable parameters',  '0 (no gradient)', f'{total_params:,}'),
    ('Training time (est.)',  '< 1 min',          '~20-40 min (GPU)'),
    ('Inference (full scene)','< 1 s (CPU)',       '~5-10 s (GPU)'),
    ('Spatial context used',  'None (per-pixel)',  '128 × 128 px window'),
]

for r in rows:
    print(f'{r[0]:<30} {r[1]:>15} {r[2]:>15}')
print('=' * 65)

In [ ]:
# ── Visual comparison bar chart ─────────────────────────────────────────────
x = np.arange(len(CLASS_NAMES))
width = 0.35

mdc_iou_vals  = [test_iou[c]      for c in CLASS_LABELS]
unet_iou_vals = [unet_test_iou[c] for c in CLASS_LABELS]

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, mdc_iou_vals,  width, label='MDC',   color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, unet_iou_vals, width, label='U-Net', color='darkorange', alpha=0.85)

ax.axhline(test_miou,      color='steelblue',  linestyle='--', linewidth=1.2, label=f'MDC mIoU  {test_miou:.3f}')
ax.axhline(unet_test_miou, color='darkorange', linestyle='--', linewidth=1.2, label=f'U-Net mIoU {unet_test_miou:.3f}')

ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('IoU'); ax.set_ylim(0, 1)
ax.set_title('Per-class IoU: MDC vs U-Net (Test Set)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(MODEL_FOLDER, 'comparison_iou_barchart.png'), dpi=150)
plt.show()

## 3.1 — Performance Analysis

### Minimum Distance Classifier (MDC)

The MDC classifies each pixel purely by its Euclidean distance to 4 class mean signatures in 17-dimensional feature space. Despite its simplicity it performs competitively on spectrally distinct classes (e.g. **Water**, which has a highly characteristic NDWI profile) and is deterministic and interpretable.

Its main weakness is **lack of spatial context**: a single bright pixel surrounded by sand is just as likely to be labelled 'Cement' as an entire city block, because the classifier cannot see its neighbourhood. This leads to salt-and-pepper noise in the predicted maps, especially in transitional zones (e.g. irrigated desert margins between Sand and Greenery).

**Computational profile** — training is instant (just computing 4 mean vectors). Inference is a vectorised distance computation: O(N × C × F) where N is pixel count, C = 4 classes, F = 17 features.

### U-Net

The U-Net sees each pixel in the context of a 128×128 receptive field. Skip connections feed high-resolution spatial detail from the encoder directly into the decoder, allowing the network to simultaneously reason at coarse semantic level (is this region urban or desert?) and fine detail (where exactly does the urban boundary lie?).

Expected gains over MDC:
- **Cement/Greenery boundary precision** — skip connections recover sharp edges lost in pooling.
- **Mixed-class transitional zones** — spatial context resolves ambiguous pixels.
- **Robustness to single-pixel noise** — a spatially smooth predicted map is enforced by the convolutional receptive field.

The cost is ~20-40 minutes of GPU training, a 31M-parameter model, and more complex hyperparameter tuning.

## 3.2 — Architectural Reasoning

| Design choice | Reasoning |
|---|---|
| 4 encoder levels (f→2f→4f→8f) | Captures features from 3-pixel texture up to 128-pixel spatial patterns |
| Skip connections | Preserve fine boundaries; critical for thin linear features like canal banks |
| BatchNorm + ReLU after every conv | Stabilises training across the wide spectral dynamic range of Sentinel-2 |
| AdamW + cosine annealing | AdamW weight-decay prevents overfitting on small datasets; cosine schedule avoids abrupt LR drops |
| ignore_index=0 in CrossEntropy | Prevents the network from being penalised for Unknown-class boundary pixels |
| Patch stride < patch size (96 < 128) | Overlapping patches at training time expose the network to more boundary configurations |
| Sliding-window inference with averaging | Avoids border artefacts; overlapping logit averaging smooths predictions near patch edges |

## 3.3 — Edge Cases & Failure Modes

**1. Spectrally ambiguous classes — Sand vs Cement.**  
Dry concrete and bright desert sand share very similar reflectance in the visible bands. Both models struggle here. The MDC relies heavily on NDBI and BSI to separate them, but these indices also respond to rocky terrain. The U-Net has additional context (e.g. regular grid patterns of roads vs. smooth dune surfaces) but requires enough training data for these patterns to be learnt.

**2. Cloud-masked pixels (NaN → 0).**  
Replacing NaN from cloud masking with 0 may introduce spurious low-reflectance pixels. A learnt imputation (e.g. filling from spatial neighbours) would be more principled but was avoided here to keep the pipeline simple and reproducible.

**3. Seasonal variability.**  
The dataset is limited to May–August 2023 (low-cloud summer). Agricultural fields in the Nile Delta undergo dramatic NDVI changes between growing seasons. A model trained only on summer imagery may misclassify bare harvested fields as Sand in winter test images.

**4. Small water bodies.**  
Narrow irrigation canals (< 10 m wide, i.e. sub-pixel at 10 m Sentinel-2 resolution) are inherently unresolvable. Neither model can detect them reliably — this is a sensor limitation rather than a modelling failure.

## 3.4 — Conclusion

The MDC is an excellent **baseline**: zero training time, fully interpretable, and surprisingly effective on spectrally distinctive classes. It should be the first thing run on any new satellite dataset to establish a lower-bound reference.

The U-Net trades interpretability and speed for substantially higher mIoU, especially in spatially complex regions. For production land-cover mapping of Egyptian terrain — where urban expansion into the desert creates intricate Sand/Cement boundaries — the U-Net is the preferred model.

A practical deployment pipeline would use both: the MDC for rapid quality-control screening of new imagery, and the U-Net for final high-confidence maps.